# Wheat Disease Classification: Vision Transformer + XGBoost

This notebook implements a two-phase training pipeline:
1. **Phase 1**: Train Vision Transformer (ViT) to extract features
2. **Phase 2**: Train XGBoost classifier on ViT logits

Dataset: RGB, Hyperspectral (HS), Multispectral (MS) images
Classes: Health, Other, Rust

## 1. Install Dependencies

In [ ]:
import subprocess
import sys

# Install required packages
packages = ['torch', 'torchvision', 'numpy', 'matplotlib', 'seaborn', 
           'scikit-learn', 'xgboost', 'tqdm', 'rasterio', 'Pillow']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

: 

## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from tqdm import tqdm
import xgboost as xgb
import os
from pathlib import Path
from torchvision import transforms, models
from PIL import Image
import warnings

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✓ All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Configuration

In [ ]:
# Configuration
config = {
    'root_dir': r'c:UsersShresthvscode2pyfilesdeepLearningwheatKaggle_Prepared',
    'modalities': ['RGB', 'HS', 'MS'],  # Load all modalities together
    'batch_size': 32,
    'vit_epochs': 30,
    'learning_rate': 0.001,
    'num_workers': 0,  # 0 for Windows
    'image_size': (224, 224),
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

# XGBoost parameters
xgb_params = {
    'objective': 'multi:softmax',
    'num_class': 3,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

print("Configuration:")
for key, value in config.items():
    if key != 'device':
        print(f"  {key}: {value}")
print(f"  device: {config['device']}")

## 4. Load Dataset

In [ ]:
class MultimodalWheatDataset(Dataset):
    """PyTorch Dataset for Multi-modal Wheat Disease Classification"""
    
    def __init__(self, root_dir, split='train', modalities=['RGB', 'HS', 'MS'], 
                 transform=None, image_size=(224, 224)):
        self.root_dir = root_dir
        self.split = split
        self.modalities = modalities
        self.transform = transform
        self.image_size = image_size
        
        self.classes = ['Health', 'Other', 'Rust']
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        # Load samples from first modality (all modalities have same samples)
        self.data_dirs = {m: os.path.join(root_dir, split, m) for m in modalities}
        
        # Debug: Check directory existence
        print(f"n[{split.upper()}] Checking directories:")
        for m in modalities:
            data_dir = self.data_dirs[m]
            exists = os.path.exists(data_dir)
            if exists:
                file_count = len([f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))])
                print(f"  {m}: {data_dir} ✓ ({file_count} files)")
            else:
                print(f"  {m}: {data_dir} ✗ (NOT FOUND)")
        
        self.samples = self._load_samples()
        
        if self.transform is None:
            self.transform = self._get_default_transforms()
    
    def _load_samples(self):
        """Load all image paths and labels"""
        samples = []
        
        # Use RGB as reference modality
        ref_dir = self.data_dirs['RGB']
        if not os.path.exists(ref_dir):
            print(f"⚠️  WARNING: Directory not found: {ref_dir}")
            return samples
        
        files = [f for f in os.listdir(ref_dir) if os.path.isfile(os.path.join(ref_dir, f))]
        print(f"  Found {len(files)} files in {ref_dir}")
        
        for filename in files:
            class_name = filename.split('_')[0]
            
            if class_name in self.classes:
                # Get paths for all modalities
                filepaths = {m: os.path.join(self.data_dirs[m], filename) for m in self.modalities}
                label = self.class_to_idx[class_name]
                samples.append((filepaths, label))
        
        print(f"  Loaded {len(samples)} valid samples")
        return samples
    
    def _get_default_transforms(self):
        """Get default transforms based on split"""
        if self.split == 'train':
            return transforms.Compose([
                transforms.Resize(self.image_size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
        else:
            return transforms.Compose([
                transforms.Resize(self.image_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
    
    def _load_rgb(self, filepath):
        """Load RGB image (PNG)"""
        return Image.open(filepath).convert('RGB')
    
    def _load_multispectral(self, filepath):
        """Load multispectral/hyperspectral image (TIF)"""
        try:
            import rasterio
            with rasterio.open(filepath) as src:
                image = src.read()  # (bands, height, width)
                image = np.transpose(image, (1, 2, 0))  # (height, width, bands)
                # Normalize to 0-1
                image = image.astype(np.float32)
                min_val = image.min()
                max_val = image.max()
                if max_val > min_val:
                    image = (image - min_val) / (max_val - min_val)
                # Take first 3 bands for compatibility
                if image.shape[2] >= 3:
                    image = image[:, :, :3]
                else:
                    image = np.repeat(image[:, :, 0:1], 3, axis=2)
                image = (image * 255).astype(np.uint8)
                return Image.fromarray(image)
        except Exception as e:
            print(f"⚠️  Error loading {filepath}: {e}")
            # Fallback: return blank image
            return Image.new('RGB', self.image_size)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        filepaths, label = self.samples[idx]
        
        # Load and stack images from all modalities
        images = []
        for modality in self.modalities:
            filepath = filepaths[modality]
            
            if modality == 'RGB':
                image = self._load_rgb(filepath)
            else:  # HS or MS
                image = self._load_multispectral(filepath)
            
            images.append(image)
        
        # Apply transform to each and stack
        if self.transform:
            images = [self.transform(img) for img in images]
        else:
            images = [transforms.ToTensor()(img) for img in images]
        
        # Stack along channel dimension: (3*3, 224, 224)
        image_tensor = torch.cat(images, dim=0)
        
        return image_tensor, label
    
    def get_class_distribution(self):
        """Get distribution of classes"""
        labels = [label for _, label in self.samples]
        distribution = {cls: labels.count(idx) for cls, idx in self.class_to_idx.items()}
        return distribution

# Load datasets with all modalities
print("n" + "="*60)
print("LOADING MULTI-MODAL DATASETS")
print("="*60)

train_dataset = MultimodalWheatDataset(
    root_dir=config['root_dir'],
    split='train',
    modalities=config['modalities'],
    image_size=config['image_size']
)

val_dataset = MultimodalWheatDataset(
    root_dir=config['root_dir'],
    split='val',
    modalities=config['modalities'],
    image_size=config['image_size']
)

print(f"n📊 Dataset Summary:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print(f"  Modalities: {', '.join(config['modalities'])}")
print(f"  Input shape per sample: (9, 224, 224)  # 3 modalities x 3 channels")
print(f"  Train class distribution: {train_dataset.get_class_distribution()}")
print(f"  Val class distribution: {val_dataset.get_class_distribution()}")

if len(val_dataset) == 0:
    print("n⚠️  WARNING: Val dataset is empty! Make sure to run cell 3.1 first!")

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers']
)

print(f"n📦 DataLoader Summary:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {config['batch_size']}")

## 5. Load ViT Model

In [ ]:
## 5.1. Create Custom ViT with Multi-modal Support

In [ ]:
def get_vit_model(num_classes=3, pretrained=True, model_name='vit_b_16'):
    """Load Vision Transformer model"""
    if model_name == 'vit_b_16':
        model = models.vit_b_16(weights='DEFAULT' if pretrained else None)
    elif model_name == 'vit_b_32':
        model = models.vit_b_32(weights='DEFAULT' if pretrained else None)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    
    # Replace classification head
    num_features = model.heads.head.in_features
    model.heads.head = nn.Linear(num_features, num_classes)
    
    return model

def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Load ViT model
print("Loading Vision Transformer model...")
vit_model = get_vit_model(num_classes=3, pretrained=True, model_name='vit_b_16')
vit_model = vit_model.to(config['device'])

print(f"nViT Model Information:")
print(f"  Model: Vision Transformer B16")
print(f"  Parameters: {count_parameters(vit_model):,}")
print(f"  Device: {config['device']}")

## 6. PHASE 1: Train ViT

In [ ]:
class MultimodalViT(nn.Module):
    """Vision Transformer adapted for multi-modal input (9 channels)"""
    
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        # Load pretrained ViT
        self.vit = models.vit_b_16(weights='DEFAULT' if pretrained else None)
        
        # Replace conv_proj to accept 9 channels instead of 3
        # Original: Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        original_conv = self.vit.conv_proj
        self.vit.conv_proj = nn.Conv2d(9, original_conv.out_channels, 
                                       kernel_size=original_conv.kernel_size,
                                       stride=original_conv.stride)
        
        # Initialize new layer with pretrained weights (average across channels)
        with torch.no_grad():
            original_weight = original_conv.weight  # (768, 3, 16, 16)
            new_weight = original_weight.repeat(1, 3, 1, 1)  # (768, 9, 16, 16)
            new_weight = new_weight / 3  # Normalize
            self.vit.conv_proj.weight = nn.Parameter(new_weight)
        
        # Replace classification head
        num_features = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Linear(num_features, num_classes)
    
    def forward(self, x):
        return self.vit(x)

def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Load ViT model
print("Loading Multi-modal Vision Transformer model...")
vit_model = MultimodalViT(num_classes=3, pretrained=True)
vit_model = vit_model.to(config['device'])

print(f"nViT Model Information:")
print(f"  Model: Vision Transformer B16 (Multi-modal)")
print(f"  Input channels: 9 (RGB + HS + MS)")
print(f"  Parameters: {count_parameters(vit_model):,}")
print(f"  Device: {config['device']}")

## 7. PHASE 2: Extract Logits from ViT

In [ ]:
def extract_logits(model, data_loader, device, phase='train'):
    """Extract logits from ViT model"""
    model.eval()
    all_logits = []
    all_labels = []
    
    pbar = tqdm(data_loader, desc=f'Extracting {phase} logits')
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            logits = model(images)
            
            all_logits.append(logits.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    logits = np.vstack(all_logits)
    labels = np.array(all_labels)
    
    print(f"{phase.capitalize()} logits shape: {logits.shape}")
    print(f"{phase.capitalize()} labels shape: {labels.shape}")
    
    return logits, labels

# Extract logits
print("n" + "="*60)
print("PHASE 2: Extracting Logits from ViT")
print("="*60 + "n")

train_logits, train_labels = extract_logits(vit_model, train_loader, config['device'], phase='train')
print()
val_logits, val_labels = extract_logits(vit_model, val_loader, config['device'], phase='val')

print(f"nLogits extracted successfully!")

## 8. PHASE 3: Train XGBoost on Logits

In [ ]:
print("n" + "="*60)
print("PHASE 3: Training XGBoost on ViT Logits")
print("="*60 + "n")

# Create XGBoost datasets
dtrain = xgb.DMatrix(train_logits, label=train_labels)
dval = xgb.DMatrix(val_logits, label=val_labels)

# Train with early stopping
evals = [(dtrain, 'train'), (dval, 'eval')]
evals_result = {}

print("Training XGBoost classifier...n")
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=500,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=20,
    verbose_eval=20
)

print(f"n✓ XGBoost training completed")
print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Best score: {xgb_model.best_score}")

## 9. PHASE 4: Evaluation

In [ ]:
print("n" + "="*60)
print("PHASE 4: Final Evaluation")
print("="*60 + "n")

# Predictions on validation set
dval_pred = xgb.DMatrix(val_logits)
predictions = xgb_model.predict(dval_pred)
predictions = predictions.astype(int)

# Calculate metrics
accuracy = accuracy_score(val_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    val_labels, predictions, average='weighted', zero_division=0
)

print("nFinal Metrics (Validation Set):")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Confusion matrix
cm = confusion_matrix(val_labels, predictions)
class_names = ['Health', 'Other', 'Rust']

print(f"nConfusion Matrix:")
print(cm)

# Per-class metrics
print(f"nPer-Class Metrics:")
precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
    val_labels, predictions, average=None, zero_division=0
)

for i, class_name in enumerate(class_names):
    print(f"  {class_name}: Precision={precision_per_class[i]:.4f}, "
          f"Recall={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}")

## 10. Visualization

In [ ]:
# Plot 1: ViT Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(vit_train_losses, marker='o', label='Train Loss')
axes[0].set_title('ViT Training Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(vit_train_accs, marker='o', label='Train Acc')
axes[1].plot(vit_val_accs, marker='s', label='Val Acc')
axes[1].set_title('ViT Training/Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('vit_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ ViT training curves saved")

In [ ]:
# Plot 2: XGBoost Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Extract metrics from evals_result
train_loss = evals_result.get('train', {}).get('mlogloss', [])
val_loss = evals_result.get('eval', {}).get('mlogloss', [])

if train_loss and val_loss:
    axes[0].plot(train_loss, marker='o', label='Train Loss')
    axes[0].plot(val_loss, marker='s', label='Val Loss')
    axes[0].axvline(xgb_model.best_iteration, color='r', linestyle='--', alpha=0.7, label='Best Iter')
    axes[0].set_title('XGBoost Training Loss', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Boosting Round')
    axes[0].set_ylabel('Loss (mlogloss)')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

# Feature importance
importance = xgb_model.get_score(importance_type='weight')
if importance:
    top_n = min(10, len(importance))
    sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:top_n]
    features = [f'Logit {i}' for i in range(len(importance))]
    
    features = [f[0] for f in sorted_importance]
    scores = [f[1] for f in sorted_importance]
    
    axes[1].barh(range(len(scores)), scores, color='steelblue')
    axes[1].set_yticks(range(len(features)))
    axes[1].set_yticklabels(features)
    axes[1].set_title('Top Feature Importance (XGBoost)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Importance Score')
    axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('xgboost_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ XGBoost results saved")

In [ ]:
# Plot 3: Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)

ax.set_title('Confusion Matrix - ViT + XGBoost', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Confusion matrix saved")

## 11. Summary

In [ ]:
print("n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)

print("n📊 Dataset Information:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print(f"  Classes: {', '.join(class_names)}")

print("n🧠 Phase 1 - Vision Transformer:")
print(f"  Epochs trained: {config['vit_epochs']}")
print(f"  Final train acc: {vit_train_accs[-1]:.4f}")
print(f"  Best val acc: {best_vit_acc:.4f}")
print(f"  Model parameters: {count_parameters(vit_model):,}")

print("n🚀 Phase 2 - XGBoost Classifier:")
print(f"  Input features: 3 (ViT logits)")
print(f"  Boosting rounds: {xgb_model.best_iteration + 1}")
print(f"  Max depth: {xgb_params['max_depth']}")
print(f"  Learning rate: {xgb_params['learning_rate']}")

print("n📈 Final Results (Validation Set):")
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

print("n✓ Training pipeline completed successfully!")
print("="*60)

## 12. Save Models (Optional)

In [ ]:
# Save ViT model
torch.save(vit_model.state_dict(), 'vit_model.pth')
print("✓ ViT model saved to 'vit_model.pth'")

# Save XGBoost model
xgb_model.save_model('xgb_model.json')
print("✓ XGBoost model saved to 'xgb_model.json'")

print("nModels saved successfully!")

## 13. Inference Example (Optional)

In [ ]:
def predict_single_image(image_path, vit_model, xgb_model, device, transform):
    """Predict class for a single image"""
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Get ViT logits
    vit_model.eval()
    with torch.no_grad():
        logits = vit_model(image_tensor).cpu().numpy()
    
    # XGBoost prediction
    dtest = xgb.DMatrix(logits)
    prediction = xgb_model.predict(dtest)[0]
    
    class_names = ['Health', 'Other', 'Rust']
    predicted_class = class_names[int(prediction)]
    
    return predicted_class, logits, prediction

# Example: predict on a random validation image
if len(val_dataset) > 0:
    sample_image, sample_label = val_dataset[0]
    sample_image_path = val_dataset.samples[0][0]
    class_names = ['Health', 'Other', 'Rust']
    
    pred_class, logits, pred_idx = predict_single_image(
        sample_image_path, vit_model, xgb_model, config['device'], val_dataset.transform
    )
    
    print(f"nInference Example:")
    print(f"  True class: {class_names[sample_label]}")
    print(f"  Predicted class: {pred_class}")
    print(f"  ViT logits: {logits[0]}")
    print(f"  Confidence: {max(logits[0]):.4f}")